# Decision Log — v3 All Features Trial (Emerging Market)

Documents the decision to include previously-excluded features and data quality fixes in a trial model iteration.

## v3 All Features Trial — Decision Log

**Date:** 27 October 2026
**Author:** AI Assistant (Genie Code) acting on user instructions
**Purpose:** Trial to see how the model improves when including features which on an actuarial basis we might decide not to include, but are including now for testing purposes.

---

### 1. Features Added (Previously Excluded)

The following 6 features were previously excluded from the v2 model (per the Features spreadsheet and DATA_CONTRACT.md) but are now included in the v3 trial:

| Feature | Lines | Previous Exclusion Reason | v3 Treatment |
|---|---|---|---|
| `discount_applied` | All | Actuarial: discount is a pricing decision, not a risk factor | Numeric |
| `written_premium` | All | Actuarial: premium charged is not a risk factor (target is expected ultimate cost) | Numeric |
| `customer_service_contacts` | All | Leakage risk: may correlate with claims experience | Numeric |
| `marketing_consent` | All | Not expected to predict claim frequency/severity | Categorical (bool) |
| `document_preference` | All | Not expected to predict claim frequency/severity | Categorical (object) |
| `vehicle_colour` | Motor only | Not expected to predict claim frequency/severity | Categorical (object) |

**Additionally** from email guidance on categorical treatment:
- `engine_size_cc` (motor) — added as categorical (was not in v2 features)
- `branch_id` (all) — added as categorical (was not in v2 features)
- `customer_since_year` (all) — added as categorical (was not in v2 features)

### 2. Data Quality Fixes (from Email Guidance)

**Source:** Email from team member providing data-side guidance for the hackathon.

#### 2a. Categorical Variable Treatment

The following variables were stored as numeric in the data but should be treated as **categorical** for the GLM (to avoid the GLM assuming linear relationships):

| Variable | Original dtype | Unique Values | v3 Treatment |
|---|---|---|---|
| `vehicle_value_group` | int64 | 20 (motor) | Converted to string → target encoded in GLM |
| `engine_size_cc` | float64 | 31 (motor) | Converted to string → target encoded in GLM |
| `branch_id` | int64 | 25 (all) | Converted to string → target encoded in GLM |
| `flood_zone` | float64 | 5 (buildings/contents) | Converted to string → target encoded in GLM |
| `customer_since_year` | int64 | 17 (all) | Converted to string → target encoded in GLM |
| `year_built` | int64 | 160 (buildings/contents) | Converted to string → target encoded in GLM |

**Note:** XGBoost handles these as either numeric or categorical without issue, but the GLM with log-link would assume a linear log-linear relationship if left as numeric.

#### 2b. Occupation Field Cleanup

The occupation field had significant data quality issues:
- **Case inconsistency:** "Retired" vs "retired", "Student" vs "student" (same occupation, different casing)
- **Trailing whitespace:** Some values had trailing spaces creating duplicate categories
- **"Self-employed" prefix:** 4,095 entries had "Self-employed " prepended to another occupation (e.g., "Self-employed teacher", "Self-employed retired", "Self-employed unemployed")
- **80 unique values** before cleanup → ~40 after cleanup

**Cleanup applied:**
1. Strip whitespace
2. Convert to lowercase
3. Remove "self-employed " prefix (e.g., "Self-employed teacher" → "teacher")
4. "Self-employed unemployed" → "unemployed" (clearly wrong)
5. "Self-employed retired" → "retired" (can't be both)

#### 2c. Campaign Code Omitted

`campaign_code` has ~80% missing data (80.6% motor, 80.7% buildings, 81.0% contents). Omitted from modelling entirely.

#### 2d. Mode Imputation for ~12% Missing Values

The following fields have approximately 12% missing data. Mode imputation applied (per email guidance) for the GLM:

| Field | Motor | Buildings | Contents |
|---|---|---|---|
| `overnight_parking` | 12.0% | — | — |
| `occupation` | 12.0% | 12.2% | 11.9% |
| `marital_status` | 12.1% | 12.0% | 12.0% |
| `soil_type` | 12.0% | 12.1% | 12.3% |
| `roof_type` | 12.1% | 11.9% | 11.9% |
| `energy_rating` | 11.8% | 11.9% | 11.7% |
| `smoke_alarms` | — | 11.9% | — |
| `security_level` | — | — | 12.3% |

### 3. Model Architecture

The v3 trial uses the **same model architecture** as v2:
- **XGBoost:** Tweedie (power=1.5), 50 trees, depth=2, min_child_weight=50, reg_alpha=2.0, reg_lambda=5.0
- **GLM:** Tweedie (power=1.5) with target encoding (SMOOTHING=50), StandardScaler on numeric features
- **Two-stage:** Logistic P(claim) × Gamma severity
- **3-way blend:** Optimized per-line weights
- **Calibration:** Per-line scaling to training mean × trend factor

### 4. Feature Sets

| Line | v2 Features | v3 Features | New Features Added |
|---|---|---|---|
| Motor | 13 | 22 | +6 excluded +3 categorical = +9 |
| Buildings | 13 | 20 | +5 excluded +2 categorical = +7 |
| Contents | 13 | 20 | +5 excluded +2 categorical = +7 |

### 5. Expected Outcomes

**Risk:** Including features like `written_premium` and `customer_service_contacts` may introduce leakage or confounding, potentially degrading model performance. The XGBoost may handle the additional features fine, but the GLM may struggle with more parameters relative to the number of claims (~6,000 motor, ~1,000 buildings, ~1,000 contents).

**Potential benefit:** Some excluded features (e.g., `vehicle_colour`, `document_preference`) may carry weak signal that improves predictions.

### 6. Files Created

| File | Location |
|---|---|
| 02_XGBoost_v3_all_features | notebooks_emerging_v3_all_features/ |
| 03_Two_Stage_GLM_Blend_v3_all_features | notebooks_emerging_v3_all_features/ |
| 05_Decision_Log_v3_all_features | notebooks_emerging_v3_all_features/ |
| EDA Emerging Market (copy) | notebooks_emerging_v3_all_features/ |
| 04_BF_Development_Analysis (copy) | notebooks_emerging_v3_all_features/ |
| xgboost_v3_all_features_results.pkl | data_emerging/proc/ |
| blend_calibrated_v3_all_features.csv | submissions_emerging_v3_all_features/ |
| v3_all_features_results.xlsx | Excel files_emerging_v3_all_features/ |

### 7. Decision Attribution

- **User decision:** Include previously-excluded features as a trial, apply email guidance on data quality
- **AI assistant execution:** Copied v2 notebooks to v3, implemented data cleaning, updated feature sets, updated output paths
- **Email source:** Team member email providing data-side guidance on categorical treatment, occupation cleanup, campaign_code omission, and mode imputation